# Đánh giá E5 + Reranker — corpus 5000 SP

Pipeline 2 giai đoạn:
1. **Bi-encoder** `e5_base_finetuned_5000` → retrieve top-**n**
2. **Cross-encoder reranker** → xếp hạng lại → lấy top-**k**

**Metrics:** Precision@k, Recall@k, **F1@k**, MRR@k, NDCG@k

**Ngưỡng (threshold):** FPR, FNR, EER, ngưỡng error rate nhỏ nhất

**Grid search:** tìm **n**, **k** tối ưu

> **Đánh giá trên `ecommerce.csv` (5000 SP):** query = `title`, corpus = `searchable_text`.  
> Chạy GPU Colab (T4). Full 5000 query × n=500 có thể ~1–2 giờ.

## 1) Cài thư viện

In [ ]:
!pip -q install "sentence-transformers>=3.0.0" "transformers>=4.40.0" "accelerate>=1.1.0" torch datasets pandas scikit-learn numpy tqdm

## 2) Setup đường dẫn

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/PhamMinhDan/llm_provider_benchmarking_ver2.git"
COLAB_REPO = Path("/content/llm_provider_benchmarking")

if not (COLAB_REPO / "embedding_project" / "data" / "ecommerce.csv").is_file():
    if COLAB_REPO.exists():
        shutil.rmtree(COLAB_REPO)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO)], check=True)

REPO_DIR = COLAB_REPO
SCRIPTS = REPO_DIR / "embedding_project" / "scripts"
sys.path.insert(0, str(SCRIPTS))

EMB_MODEL = REPO_DIR / "embedding_project" / "models" / "e5_base_finetuned_5000"
RERANKER = REPO_DIR / "embedding_project" / "models" / "reranker"
EVAL_CSV = REPO_DIR / "embedding_project" / "data" / "ecommerce.csv"
OUTPUT_JSON = REPO_DIR / "embedding_project" / "outputs" / "evaluation" / "reranker_pipeline_eval.json"

for p in [EMB_MODEL, RERANKER, EVAL_CSV]:
    print('OK' if p.exists() else 'MISSING', p)

## 3) Chạy đánh giá đầy đủ

Smoke test: đặt `MAX_QUERIES = 20`. Full: `MAX_QUERIES = None`.

In [ ]:
import torch

MAX_QUERIES = None  # None = full 5000 sản phẩm trong ecommerce.csv
QUERY_COL = "title"
N_VALUES = [20, 50, 100, 200, 500]
K_VALUES = [5, 10, 20]
EVAL_K = 10

cmd = [
    sys.executable,
    str(SCRIPTS / "evaluate_reranker_pipeline.py"),
    "--embedding-model", str(EMB_MODEL),
    "--reranker-model", str(RERANKER),
    "--eval-csv", str(EVAL_CSV),
    "--query-col", QUERY_COL,
    "--output", str(OUTPUT_JSON),
    "--eval-k", str(EVAL_K),
    "--n-values", *[str(n) for n in N_VALUES],
    "--k-values", *[str(k) for k in K_VALUES],
]
if MAX_QUERIES:
    cmd.extend(["--max-queries", str(MAX_QUERIES)])

print('CUDA:', torch.cuda.is_available())
!{" ".join(cmd)}

## 4) Bảng kết quả & biểu đồ ngưỡng

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

result = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
eval_k = 10

bi = result[f"bi_encoder_only@{eval_k}"]
rk = result[f"reranker_best_n@{eval_k}"]["metrics"]
opt = result["optimal_n_k"][f"by_max_f1_at_k{eval_k}"]
thr = result["threshold_analysis"]

summary = pd.DataFrame([
    {"stage": "Bi-encoder only", **{k: bi[k] for k in bi if k.startswith(("Precision", "Recall", "F1"))}},
    {"stage": f"Reranker (n={result['reranker_best_n@10']['n']})", **{k: rk[k] for k in rk if k.startswith(("Precision", "Recall", "F1"))}},
])
print("=== So sánh @10 ===")
display(summary)

print("\n=== n, k tối ưu (max F1@10) ===")
print(json.dumps(opt, ensure_ascii=False, indent=2))

print("\n=== Ngưỡng reranker ===")
print("EER (FPR ≈ FNR):", json.dumps(thr["EER"], indent=2))
print("Min error rate:", json.dumps(thr["min_error_rate"], indent=2))

curve = pd.DataFrame(result["threshold_curve"])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(curve["threshold"], curve["FPR"], label="FPR")
ax[0].plot(curve["threshold"], curve["FNR"], label="FNR")
ax[0].axvline(thr["EER"]["threshold"], color="gray", ls="--", label=f"EER τ={thr['EER']['threshold']:.3f}")
ax[0].set_xlabel("threshold"); ax[0].set_ylabel("rate"); ax[0].legend(); ax[0].set_title("FPR / FNR vs threshold")
ax[1].plot(curve["threshold"], curve["error_rate"], color="crimson")
ax[1].axvline(thr["min_error_rate"]["threshold"], color="gray", ls="--")
ax[1].set_xlabel("threshold"); ax[1].set_ylabel("error rate"); ax[1].set_title("Error rate vs threshold")
plt.tight_layout(); plt.show()

grid = pd.DataFrame(result["grid_search_n_k"])
pivot = grid.pivot(index="n", columns="k", values="F1@10")
print("\n=== Heatmap F1@10 (n × k) ===")
display(pivot)

## 5) Giải thích cho báo cáo GVHD

- **n**: số ứng viên bi-encoder lấy ra (recall stage-1)
- **k**: số kết quả cuối sau rerank (thường k=10)
- **F1@10** = 2·P·R/(P+R) — metric cân bằng hơn P@10 đơn lẻ khi 1 nhãn/query
- **EER**: ngưỡng τ sao cho FPR(τ) ≈ FNR(τ) — điểm cân bằng false alarm / miss
- **Min error**: τ làm (FP+FN) nhỏ nhất trên cặp (query, passage) có nhãn